In [1]:
WIKIMEDIA_EVENTSTREAMS_base_url = "https://stream.wikimedia.org/"
WIKIMEDIA_EVENTSTREAMS_TOPICS_METADATA = {
    "recentchange": { "path": "/v2/stream/recentchange", "primary_key": ("id",) }
}

USER_AGENT = "TEST"

BASE_DIR = ""
QUEUES_SUBDIR = "queues"

TOPIC_NEW_QUEUE_FILE_SECONDS_THRESHOLD = 2
LOAD_INTO_DB_EVERY_SECONDS = 15

In [2]:
import asyncio
from asyncio import IncompleteReadError
import aiofiles
from aiofiles.threadpool.binary import AsyncBufferedIOBase
import aiohttp
from aiohttp import ClientPayloadError
from pathlib import Path
from typing import Iterable, Tuple, Coroutine
from datetime import datetime
import json
from json import JSONDecodeError
import sseclient
import dlt
from dlt.sources.filesystem import filesystem
from dlt.extract.resource import DltResource
from dlt.common.storages.fsspec_filesystem import FileItemDict
import time
import threading
from threading import Thread

c:\ProgramData\uv\misc\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.3.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


In [3]:
def create_local_layers_dirs(base_dir: str) -> Path:
    data_path = Path(base_dir) / "data"
    data_path.mkdir(exist_ok=True)

    for layer in ["queues", "csv"]:
        layer_path = data_path / layer
        layer_path.mkdir(exist_ok=True)

    return data_path

def create_local_queues_paths(queues_base_path: Path, topics: Iterable[str]) -> dict[str, Path]:
    all_queues_basepaths = dict()

    for topic in topics:
        all_queues_basepaths[topic] = queues_base_path / f"{topic}"
        all_queues_basepaths[topic].mkdir(exist_ok=True)

    return all_queues_basepaths

In [4]:
async def get_new_topic_queue_file(
    topic: str, local_queues_basepaths: dict[str, Path]
) -> Tuple[datetime, AsyncBufferedIOBase]:
    topic_last_queue_file_datetime = datetime.now()
    topic_last_queue_full_path = (
        local_queues_basepaths[topic]
        / f"{str(topic_last_queue_file_datetime.timestamp())}.bin"
    )
    topic_last_queue_fp = await aiofiles.open(topic_last_queue_full_path, "wb")

    return topic_last_queue_file_datetime, topic_last_queue_fp


async def write_topic_local_queue(
    topic: str,
    base_url: str,
    sse_topics_metadata: dict[str, dict[str, str]],
    local_queues_basepaths: dict[str, Path],
):
    last_queue_file_datetime: datetime | None = None
    last_queue_file: AsyncBufferedIOBase | None = None

    timeout = aiohttp.ClientTimeout(total=None, sock_read=None)
    
    while True:
        try:
            async with aiohttp.ClientSession(timeout=timeout) as session, session.get(
                base_url + "/" + sse_topics_metadata[topic]["path"],
                headers={"User-Agent": USER_AGENT},
            ) as resp:
                while True:
                    try:
                        current_contents = await resp.content.readuntil(separator=b"\n\n")
                    except IncompleteReadError:
                        print("ERROR: streaming request error, incomplete read")
                        break

                    if not last_queue_file or not last_queue_file_datetime:
                        last_queue_file_datetime, last_queue_file = (
                            await get_new_topic_queue_file(topic, local_queues_basepaths)
                        )

                    if (
                        datetime.now() - last_queue_file_datetime
                    ).seconds > TOPIC_NEW_QUEUE_FILE_SECONDS_THRESHOLD:
                        await last_queue_file.close()
                        last_queue_file_datetime, last_queue_file = (
                            await get_new_topic_queue_file(topic, local_queues_basepaths)
                        )

                    await last_queue_file.write(current_contents)

        except ClientPayloadError:
            print(f"ERROR: Connection dropped randomly on topic {topic}, trying again...")


async def write_all_topics_local_queues(
    base_url: str,
    sse_topics_metadata: dict[str, dict[str, str]],
    local_queues_basepaths: dict[str, Path],
):
    queues_awaitables_list: list[Coroutine] = [
        write_topic_local_queue(
            topic, base_url, sse_topics_metadata, local_queues_basepaths
        )
        for topic in local_queues_basepaths.keys()
    ]

    await asyncio.gather(*queues_awaitables_list)

In [5]:
@dlt.transformer(write_disposition="append")
def read_binary_topics(
    event_files: list[FileItemDict], topic: str, primary_key: Iterable[str]
) -> list[dict]:
    records_list: list[dict] = []

    seen_records = 0
    seen_missing_primary_key_records = 0
    seen_incomplete_records = 0

    for event_file_dict in event_files:
        with event_file_dict.open("rb") as event_file:
            events = sseclient.SSEClient(event_file).events()

            while True:
                try:
                    event = next(events)
                except UnicodeDecodeError as error:
                    print(
                        f"ERROR: Decoding UTF-8 failed with file {event_file_dict["relative_path"]}"
                    )
                    print(error)
                    continue
                except StopIteration:
                    break

                seen_records += 1

                try:
                    event_data = json.loads(event.data)
                    event_primary_key_missing = False

                    for primary_key_part in primary_key:
                        if not primary_key_part in event_data:
                            event_primary_key_missing = True
                            seen_missing_primary_key_records += 1

                            continue

                    if not event_primary_key_missing:
                        records_list.append(event_data)

                except JSONDecodeError:
                    seen_incomplete_records += 1

    print(
        f"WARNING: Out of {seen_records} total records, \n\t{seen_missing_primary_key_records} marked as missing primary key,\n\t{seen_incomplete_records} marked as incomplete."
    )

    return records_list


def get_filesystem_resources(
    sse_topics_metadata: dict[str, dict[str, str]],
    local_queues_basepaths: dict[str, Path],
) -> dict[str, DltResource]:
    filesystem_resources: dict[str, DltResource] = dict()

    for topic in local_queues_basepaths.keys():
        filesystem_resources[topic] = (
            filesystem(
                bucket_url=str(local_queues_basepaths[topic]),
                incremental=dlt.sources.incremental("modification_date")
            )
            | read_binary_topics(topic, sse_topics_metadata[topic]["primary_key"])
        ).apply_hints(merge_key=sse_topics_metadata[topic]["primary_key"])

    return filesystem_resources

In [6]:
stop_event = threading.Event()

async def listen_to_sse_topics(
    base_url: str,
    sse_topics_metadata: dict[str, dict[str, str]],
    queues_base_paths: dict[str, Path],
):
    while not stop_event.is_set():
        await write_all_topics_local_queues(
            base_url, sse_topics_metadata, queues_base_paths
        )


def schedule_data_transformation(
    sse_topics_metadata: dict[str, dict[str, str]],
    queues_base_paths: dict[str, Path],
    warehouse_path: Path,
    seconds: int,
):
    while not stop_event.is_set():
        filesystem_resources = get_filesystem_resources(
            sse_topics_metadata, queues_base_paths
        )

        pipeline = dlt.pipeline(
            pipeline_name="bronze_stream",
            pipelines_dir=".dlt",
            destination=dlt.destinations.duckdb(str(warehouse_path)),
            dataset_name="bronze"
        )

        for topic, filesystem_resource in filesystem_resources.items():
            pipeline.run(
                filesystem_resource, table_name=topic
            )

        time.sleep(seconds)

In [7]:
data_path = create_local_layers_dirs(BASE_DIR)
all_queues_basepaths = create_local_queues_paths(
    data_path / "queues", WIKIMEDIA_EVENTSTREAMS_TOPICS_METADATA.keys()
)

sse_listener_thread = Thread(
    target=asyncio.run,
    args=(
        listen_to_sse_topics(
            WIKIMEDIA_EVENTSTREAMS_base_url,
            WIKIMEDIA_EVENTSTREAMS_TOPICS_METADATA,
            all_queues_basepaths,
        ),
    ),
    daemon=True,
)

bronze_writer_thread = Thread(
    target=schedule_data_transformation,
    args=(
        WIKIMEDIA_EVENTSTREAMS_TOPICS_METADATA,
        all_queues_basepaths,
        data_path / "warehouse.duckdb",
        LOAD_INTO_DB_EVERY_SECONDS,
    ),
    daemon=True,
)

stop_event.clear()
sse_listener_thread.start()
bronze_writer_thread.start()

In [ ]:
# stop_event.set()